In [8]:
# 函数调用
from langchain_ollama import ChatOllama, OllamaEmbeddings
from openai import base_url

In [9]:
# 定义一个获取当前天气的函数
import json

def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in given location"""
    weather_info = {
        "location" : location,
        "temperature" : "72",
        "unit" : unit,
        "forecast" : ["sunny", "windy"],
    }
    return json.dumps(weather_info)
    # 目前通过硬编码返回信息

In [10]:
# 传递消息给语言模型，OpenAI新增了一个名为function说的参数 ，拆的你一组函数定义
# 其中包含几个不同参数的JSON对象
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
]

In [11]:
# 创建消息列表
messages = [
    {
        "role": "user",
        "content" : "What's the weather like in Boston?"
    }
]

In [12]:
from openai import OpenAI

In [13]:
# 此处为1.0版本以上的openai包，因此需要创建client对象
client = OpenAI(
    base_url="http://localhost:11434/v1", # 本地的ollama服务url格式
    api_key="ollama" # ollama不需要api_key
)

In [15]:
response = client.chat.completions.create(
    # 需要启动本地ollama服务，可在任务管理器当中打开默认启动
    model="qwen3.5:2b",
    messages=messages,
    tools=[{"type":"function","function":functions[0]}]
)

In [16]:
print(response)

ChatCompletion(id='chatcmpl-476', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_b8fam89p', function=Function(arguments='{"location":"Boston, MA","unit":"celsius"}', name='get_current_weather'), type='function', index=0)], reasoning='The user is asking about the weather in Boston. I need to use the get_current_weather function to get this information. I should use "Boston, MA" as the location since Boston is in Massachusetts.\n\nLet me check the requirements:\n- location is required: "Boston, MA" (city and state format)\n- unit is optional (though I\'ll use Celsius as it\'s more common internationally, but I should check what the function accepts according to the enum)\n\nLooking at the function definition:\n- location: required - "The city and state, e.g. San Francisco, CA"\n- unit: o

In [29]:
response_reasoning = response.choices[0].message # 打印message输出
# 查看返回的消息
response_reasoning

ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_b8fam89p', function=Function(arguments='{"location":"Boston, MA","unit":"celsius"}', name='get_current_weather'), type='function', index=0)], reasoning='The user is asking about the weather in Boston. I need to use the get_current_weather function to get this information. I should use "Boston, MA" as the location since Boston is in Massachusetts.\n\nLet me check the requirements:\n- location is required: "Boston, MA" (city and state format)\n- unit is optional (though I\'ll use Celsius as it\'s more common internationally, but I should check what the function accepts according to the enum)\n\nLooking at the function definition:\n- location: required - "The city and state, e.g. San Francisco, CA"\n- unit: optional (not required according to the required field, but it has a list)\n\nI\'ll call the function with Bost

In [33]:
response_reasoning.reasoning

'The user is asking about the weather in Boston. I need to use the get_current_weather function to get this information. I should use "Boston, MA" as the location since Boston is in Massachusetts.\n\nLet me check the requirements:\n- location is required: "Boston, MA" (city and state format)\n- unit is optional (though I\'ll use Celsius as it\'s more common internationally, but I should check what the function accepts according to the enum)\n\nLooking at the function definition:\n- location: required - "The city and state, e.g. San Francisco, CA"\n- unit: optional (not required according to the required field, but it has a list)\n\nI\'ll call the function with Boston, MA as the location.'

In [46]:
response_reasoning.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_b8fam89p', function=Function(arguments='{"location":"Boston, MA","unit":"celsius"}', name='get_current_weather'), type='function', index=0)]

In [57]:
response_reasoning.tool_calls[0].function.arguments

'{"location":"Boston, MA","unit":"celsius"}'

In [58]:
json.loads(response_reasoning.tool_calls[0].function.arguments)

{'location': 'Boston, MA', 'unit': 'celsius'}

In [63]:
args = json.loads(response_reasoning.tool_calls[0].function.arguments)

In [65]:
# OpenAI的函数调用并不会直接让大模型调用函数，而是告知应该调用哪个函数，包括函数名称和参数
get_current_weather(args)

'{"location": {"location": "Boston, MA", "unit": "celsius"}, "temperature": "72", "unit": "fahrenheit", "forecast": ["sunny", "windy"]}'

In [66]:
# 查看与输入无关的测试结果
messages = [
    {
        "role": "user",
        "content": "hi",
    }
]

In [75]:
# 重新创建客户端对象
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama_model"
)

In [76]:
response = client.chat.completions.create(
    model="qwen3.5:2b",
    messages=messages,
    functions=functions,
)

In [81]:
print(response)

ChatCompletion(id='chatcmpl-557', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I don\'t have access to real-time weather data, so I can\'t tell you the specific condition for today in Boston.\n\nHowever, Boston\'s weather is generally characterized by the following:\n\n*   **Seasons:** Boston is in the North Temperate climate zone. It is typically cold (chilly) during the winter months and milder in the warmer months (spring and autumn).\n*   **Oceanic Influence:** Because it lies on the East Coast, Boston is prone to heavy fog, especially in the mornings during autumn and winter.\n*   **Humidity:** Summers can be humid and warm due to the proximity to the ocean and heatwaves, while winter is known for rapid temperature swings.\n\n**To get today\'s specific forecast:**\n\n1.  **Google Search:** Look for the "Boston Weather" link in search results.\n2.  **Weather Apps:** Check a weather app like Apple Weather or Weather.com (check 